In [ ]:
"""Build, validate, and idempotently merge the complete Gold model."""


In [ ]:
import json
import sys
import uuid
from pathlib import Path


def _add_project_src() -> None:
    cwd = Path.cwd()
    for candidate in (cwd / "src", cwd.parent / "src"):
        if (candidate / "carewatch").is_dir():
            value = str(candidate)
            if value not in sys.path:
                sys.path.insert(0, value)
            return
    raise RuntimeError("Could not locate src/carewatch from the Databricks Git folder")


_add_project_src()

from carewatch.audit import audited  # noqa: E402
from carewatch.config import qualified_name  # noqa: E402
from carewatch.gold import (  # noqa: E402
    GOLD_SCHEMAS,
    refresh_gold,
    successful_silver_batches_for_pipeline_run,
)


In [ ]:
dbutils.widgets.text("catalog", "carewatch")
dbutils.widgets.text("schema", "pipeline")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("pipeline_run_id", "")

PARAMETERS = {
    name: dbutils.widgets.get(name).strip()
    for name in ("catalog", "schema", "load_type", "pipeline_run_id")
}


In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

catalog = PARAMETERS["catalog"]
schema_name = PARAMETERS["schema"]
load_type = PARAMETERS["load_type"]
pipeline_run_id = PARAMETERS["pipeline_run_id"] or None
run_id = str(uuid.uuid4())
log_table = qualified_name(catalog, schema_name, "pipeline_execution_logs")

silver_batch_ids = successful_silver_batches_for_pipeline_run(
    spark, log_table, pipeline_run_id
)
batch_id = (
    f"gold:{pipeline_run_id}" if pipeline_run_id else f"gold:manual:{run_id}"
)
parameter = json.dumps(
    {
        "catalog": catalog,
        "schema": schema_name,
        "pipeline_run_id": pipeline_run_id,
        "silver_batch_ids": list(silver_batch_ids),
        "strategy": "complete_current_silver_state",
        "targets": sorted(GOLD_SCHEMAS),
    },
    separators=(",", ":"),
    sort_keys=True,
)

with audited(
    spark,
    log_table,
    run_id,
    "Silver-to-Gold",
    "all",
    load_type,
    parameter,
    batch_id,
    pipeline_run_id=pipeline_run_id,
) as record:
    result = refresh_gold(
        spark,
        catalog,
        schema_name,
        audit_record=record,
    )


In [ ]:
from pyspark.sql.types import LongType, StringType, StructField, StructType


METRIC_SCHEMA = StructType(
    [
        StructField("table", StringType(), False),
        StructField("rows_inserted", LongType(), False),
        StructField("rows_updated", LongType(), False),
        StructField("rows_deleted", LongType(), False),
    ]
)
COUNT_SCHEMA = StructType(
    [
        StructField("table", StringType(), False),
        StructField("rows", LongType(), False),
    ]
)

display(spark.createDataFrame(list(result.target_metrics), METRIC_SCHEMA).orderBy("table"))
display(spark.createDataFrame(list(result.table_counts), COUNT_SCHEMA).orderBy("table"))
print(
    f"Gold refresh succeeded: rows_read={result.rows_read}, "
    f"rows_aggregated={result.rows_aggregated}, "
    f"inserted={result.rows_inserted}, updated={result.rows_updated}, "
    f"deleted={result.rows_deleted}"
)
